# Final Evaluation

## Objective

The objective of this notebook is to evaluate the complete
Personalized Healthcare & Medicine Recommendation System.

The evaluation covers:

- Disease prediction model performance
- Disease-to-medication mapping coverage
- Recommendation pipeline verification
- System limitations

In [1]:
# Load dataset and trained model

import pandas as pd
import joblib

# Load Training dataset
training_df = pd.read_csv("../data/Training.csv")

# Load trained disease prediction model
svc_model = joblib.load("../models/best_model.pkl")
disease_encoder = joblib.load("../models/disease_encoder.pkl")

print("Training dataset shape:", training_df.shape)
print("Model loaded successfully.")
print("Number of disease classes:", len(disease_encoder.classes_))

Training dataset shape: (4920, 133)
Model loaded successfully.
Number of disease classes: 41


In [2]:
# Separate features and target

X = training_df.drop(columns=["prognosis"])
y = training_df["prognosis"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Number of features:", X.shape[1])
print("Number of classes:", y.nunique())

Feature shape: (4920, 132)
Target shape: (4920,)
Number of features: 132
Number of classes: 41


In [3]:
# Encode the actual disease labels

y_encoded = disease_encoder.transform(y)

print("Encoded target shape:", y_encoded.shape)
print("First 10 encoded labels:", y_encoded[:10])

Encoded target shape: (4920,)
First 10 encoded labels: [15 15 15 15 15 15 15 15 15 15]


In [4]:
# Create training and test sets for evaluation

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 3936
Testing samples: 984


In [5]:
# Generate predictions on the test set

y_pred = svc_model.predict(X_test)

print("Prediction completed.")
print("Number of predictions:", len(y_pred))

Prediction completed.
Number of predictions: 984


In [6]:
# Calculate disease prediction accuracy

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 1.0000
Test Accuracy: 100.00%


In [7]:
# Generate classification report

from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    target_names=disease_encoder.classes_
)

print(report)

                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        24
                                   AIDS       1.00      1.00      1.00        24
                                   Acne       1.00      1.00      1.00        24
                    Alcoholic hepatitis       1.00      1.00      1.00        24
                                Allergy       1.00      1.00      1.00        24
                              Arthritis       1.00      1.00      1.00        24
                       Bronchial Asthma       1.00      1.00      1.00        24
                   Cervical spondylosis       1.00      1.00      1.00        24
                            Chicken pox       1.00      1.00      1.00        24
                    Chronic cholestasis       1.00      1.00      1.00        24
                            Common Cold       1.00      1.00      1.00        24
                           

In [8]:
# Check disease-to-medication coverage

medications_df = pd.read_csv("../data/medications.csv")

# Normalize disease names
medication_diseases = set(
    medications_df["Disease"].astype(str).str.strip()
)

training_diseases = set(
    training_df["prognosis"].astype(str).str.strip()
)

missing_diseases = training_diseases - medication_diseases

print("Total diseases in prediction dataset:", len(training_diseases))
print("Total diseases in medication dataset:", len(medication_diseases))
print("Diseases without medication information:", len(missing_diseases))
print("Missing diseases:", sorted(missing_diseases))

Total diseases in prediction dataset: 41
Total diseases in medication dataset: 41
Diseases without medication information: 2
Missing diseases: ['(vertigo) Paroymsal  Positional Vertigo', 'Peptic ulcer diseae']


In [9]:
# Check disease-to-medication coverage after name normalization

medications_df = pd.read_csv("../data/medications.csv")

# Normalize disease names
medication_diseases = set(
    medications_df["Disease"].astype(str).str.strip()
)

training_diseases = set(
    training_df["prognosis"].astype(str).str.strip()
)

# Correct known naming differences
disease_mapping = {
    "Peptic ulcer diseae": "Peptic ulcer disease",
    "(vertigo) Paroymsal  Positional Vertigo":
        "(vertigo) Paroymsal Positional Vertigo"
}

normalized_training_diseases = {
    disease_mapping.get(disease, disease)
    for disease in training_diseases
}

missing_diseases = (
    normalized_training_diseases - medication_diseases
)

print("Total diseases in prediction dataset:", len(training_diseases))
print("Total diseases in medication dataset:", len(medication_diseases))
print("Diseases without medication information:", len(missing_diseases))
print("Missing diseases:", sorted(missing_diseases))

Total diseases in prediction dataset: 41
Total diseases in medication dataset: 41
Diseases without medication information: 0
Missing diseases: []


In [10]:
# Test the complete prediction-to-medication pipeline

# Select one test patient
sample_index = 0

sample_symptoms = X_test.iloc[[sample_index]]

# Predict encoded disease
predicted_encoded = svc_model.predict(sample_symptoms)[0]

# Convert encoded prediction back to disease name
predicted_disease = disease_encoder.inverse_transform(
    [predicted_encoded]
)[0]

# Find medication information
medication_result = medications_df[
    medications_df["Disease"].astype(str).str.strip()
    == predicted_disease.strip()
]

print("Predicted disease:", predicted_disease)

if not medication_result.empty:
    print("Medication information:")
    print(medication_result["Medication"].iloc[0])
else:
    print("No medication information found.")

Predicted disease: Hypertension 
Medication information:
['Antihypertensive medications', 'Diuretics', 'Beta-blockers', 'ACE inhibitors', 'Calcium channel blockers']


## Evaluation Summary

The final evaluation confirms that the disease prediction model
achieved 100% accuracy on the provided test split of 984 records.

The classification report shows precision, recall, and F1-score of
1.00 for all 41 disease classes in this evaluation.

The disease-to-medication mapping covers all 41 disease classes after
normalizing minor naming differences between the datasets.

The complete pipeline was also tested successfully:

Symptoms → Disease Prediction → Medication Information

The system therefore demonstrates a working end-to-end prediction
and medication information retrieval pipeline on the provided datasets.

## Limitations

- The model was evaluated only on the provided dataset.
- The reported 100% accuracy represents performance on the evaluation
  test split and does not represent real-world medical diagnostic accuracy.
- The medication dataset provides disease-wise medication information;
  it does not contain patient-level treatment outcomes.
- Therefore, the system retrieves medication information associated with
  the predicted disease rather than learning the optimal medication for
  an individual patient.
- The system should be treated as an educational and informational
  prototype, not as a substitute for professional medical advice.

## Conclusion

The Personalized Healthcare & Medicine Recommendation System
successfully demonstrates an end-to-end machine learning pipeline.

The system:

- Predicts diseases from symptom features using an SVC model.
- Achieved 100% accuracy on the provided evaluation test split.
- Provides precision, recall, and F1-score of 1.00 for all 41 disease classes
  in the evaluation.
- Connects the predicted disease with disease-wise medication information.
- Successfully verified the complete prediction-to-medication pipeline.

The project demonstrates how machine learning can be combined with
structured healthcare data to build an educational personalized
recommendation prototype.